In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

In [ ]:
## Pydantic
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title : str=Field(...,description="The title of the movie")
    year : int=Field(...,description="The year of release")
    director : str=Field(...,description="The name of director")
    rating : float=Field(...,description="IMDB rating of movie")

model_with_structure = model.with_structured_output(Movie,include_raw=False)
response = model_with_structure.invoke("Description about lala land")
response

Movie(title='La La Land', year=2016, director='Damien Chazelle', rating=8.0)

In [6]:
## pydantic -> Nested Structue
from pydantic import BaseModel,Field
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float|None = Field(None,description="In million dollars")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Description about chak-de India")
response

MovieDetails(title='Chak De! India', year=2007, cast=[Actor(name='Shah Rukh Khan', role='Coach Kabir Khan'), Actor(name='Fardeen Khan', role='Raju'), Actor(name='Raima Sen', role='Preeti')], genres=['Sports', 'Drama'], budget=10.0)

In [11]:
### TypeDict
from typing_extensions import TypedDict,Annotated

class Movie(TypedDict):
    title : Annotated[str,...,"The title of the movie"]
    year : Annotated[int,...,"The year of release"]
    director : Annotated[str,...,"The name of director"]
    rating : Annotated[float,...,"IMDB rating of movie"]

model_with_structure = model.with_structured_output(Movie,include_raw=False)
response = model_with_structure.invoke("Description about lala land")
response


{'director': 'Damien Chazelle',
 'rating': 8,
 'title': 'La La Land',
 'year': 2016}

In [14]:
### DataClass + Agent_implementation :-
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name:str
    email:str
    phone:str

agent = create_agent(
    model = 'groq:llama-3.1-8b-instant',
    response_format = ContactInfo
)
result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')